# MicroGuard — Resume-Safe Training
**Trains one model at a time. Auto-saves to Google Drive. Skips completed models.**

Just click **Runtime → Run All**. If disconnected, reconnect and Run All again.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# ============================================================
# Cell 1: Setup + Mount Drive
# ============================================================
!pip install -q torch transformers datasets peft accelerate bitsandbytes
!pip install -q scikit-learn scipy pandas numpy tqdm evaluate

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/MicroGuard'
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Check what's already done
done = []
for m in ['smollm360m', 'qwen05b', 'tinyllama', 'roberta_base', 'roberta_large', 'nli_deberta']:
    for path in [f'{DRIVE_DIR}/results/train_{m}.json', f'{DRIVE_DIR}/results/eval_{m}.json']:
        if os.path.exists(path):
            done.append(m)
            break
print(f'Already completed: {done if done else "none"}')
print('Setup complete!')

In [ ]:
# ============================================================
# Cell 2: Load or download dataset
# ============================================================
import json, hashlib
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from sklearn.model_selection import train_test_split
from collections import Counter

SEED = 42
np.random.seed(SEED)

# Try loading from Drive first (saves ~5 min on reconnect)
if os.path.exists(f'{DRIVE_DIR}/data_combined/dataset_info.json'):
    print('Loading dataset from Drive (cached)...')
    ds = load_from_disk(f'{DRIVE_DIR}/data_combined')
    print(f'Loaded: Train={len(ds["train"])}, Val={len(ds["validation"])}, Test={len(ds["test"])}')
else:
    print('Downloading datasets (first time only)...')
    
    # --- RAGBench ---
    RAGBENCH_SUBSETS = ['covidqa','cuad','delucionqa','emanual','expertqa','finqa','hagrid','hotpotqa','msmarco','pubmedqa','tatqa','techqa']
    ragbench_records = {'train':[],'validation':[],'test':[]}
    for subset in RAGBENCH_SUBSETS:
        try:
            print(f'  RAGBench/{subset}...')
            sub_ds = load_dataset('galileo-ai/ragbench', subset)
            for split in sub_ds:
                for ex in sub_ds[split]:
                    context = '\n\n'.join(ex['documents']) if isinstance(ex['documents'],list) else str(ex['documents'])
                    label = 'faithful' if ex['adherence_score'] else 'unfaithful'
                    ragbench_records[split].append({'query':ex['question'],'context':context,'answer':ex['response'],'label':label,'source':f'ragbench_{subset}'})
        except Exception as e:
            print(f'  Failed {subset}: {e}')
    print(f'RAGBench: {sum(len(v) for v in ragbench_records.values())} examples')

    # --- RAGTruth ---
    ragtruth_records = {'train':[],'validation':[],'test':[]}
    try:
        print('  RAGTruth...')
        rt_ds = load_dataset('wandb/RAGTruth-processed')
        for split in rt_ds:
            for ex in rt_ds[split]:
                lp = ex['hallucination_labels_processed']
                if isinstance(lp,str): lp = json.loads(lp)
                has_h = lp.get('evident_conflict',0)>0 or lp.get('baseless_info',0)>0
                label = 'unfaithful' if has_h else 'faithful'
                ctx = ex.get('context') or ''
                ans = ex.get('output') or ''
                if not ctx or not ans: continue
                t = 'test' if split=='test' else 'train'
                ragtruth_records[t].append({'query':ex.get('query',''),'context':ctx,'answer':ans,'label':label,'source':f"ragtruth_{ex.get('task_type','unknown')}"})
        if len(ragtruth_records['train'])>100:
            labels=[r['label'] for r in ragtruth_records['train']]
            ti,vi=train_test_split(range(len(ragtruth_records['train'])),test_size=0.1,stratify=labels,random_state=SEED)
            ragtruth_records['validation']=[ragtruth_records['train'][i] for i in vi]
            ragtruth_records['train']=[ragtruth_records['train'][i] for i in ti]
        print(f'RAGTruth: {sum(len(v) for v in ragtruth_records.values())} examples')
    except Exception as e:
        print(f'RAGTruth failed: {e}')

    # --- HaluBench ---
    halubench_records = {'train':[],'validation':[],'test':[]}
    try:
        print('  HaluBench...')
        hb_ds = load_dataset('PatronusAI/HaluBench')
        all_halu = []
        for ex in hb_ds['test']:
            label='faithful' if ex['label']=='PASS' else 'unfaithful'
            ctx=ex.get('passage',''); ans=str(ex.get('answer',''))
            if not ctx or not ans: continue
            all_halu.append({'query':ex.get('question',''),'context':ctx,'answer':ans,'label':label,'source':f"halubench_{ex.get('source_ds','unknown')}"})
        hl=[r['label'] for r in all_halu]
        tri,tei=train_test_split(range(len(all_halu)),test_size=0.2,stratify=hl,random_state=SEED)
        tel=[hl[i] for i in tei]
        vi2,ti2=train_test_split(tei,test_size=0.5,stratify=tel,random_state=SEED)
        halubench_records['train']=[all_halu[i] for i in tri]
        halubench_records['validation']=[all_halu[i] for i in vi2]
        halubench_records['test']=[all_halu[i] for i in ti2]
        print(f'HaluBench: {sum(len(v) for v in halubench_records.values())} examples')
    except Exception as e:
        print(f'HaluBench failed: {e}')

    # Combine
    def dedup(recs):
        seen=set(); out=[]
        for r in recs:
            k=hashlib.md5((r['context'][:500]+r['answer'][:500]).encode()).hexdigest()
            if k not in seen: seen.add(k); out.append(r)
        return out
    combined={s:dedup(ragbench_records.get(s,[])+ragtruth_records.get(s,[])+halubench_records.get(s,[])) for s in ['train','validation','test']}
    ds=DatasetDict({s:Dataset.from_list(combined[s]) for s in combined})
    ds.save_to_disk(f'{DRIVE_DIR}/data_combined')
    for s in ds: print(f'  {s}: {len(ds[s])} — {dict(Counter(ds[s]["label"]))}')
    stats={'total':sum(len(ds[s]) for s in ds),'splits':{s:{'total':len(ds[s]),**dict(Counter(ds[s]['label']))} for s in ds}}
    with open(f'{DRIVE_DIR}/results/dataset_stats.json','w') as f: json.dump(stats,f,indent=2)

print(f'Dataset ready: {len(ds["train"])} train, {len(ds["test"])} test')

In [ ]:
# ============================================================
# Cell 3: Shared config + helper functions (IMPROVED)
# ============================================================
import gc, time
from datetime import datetime
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
    RobertaForSequenceClassification, RobertaTokenizer,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, cohen_kappa_score
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
MAX_SEQ_LEN = 512  # Increased from 512 for more context
MAX_TRAIN_SAMPLES = 40000  # Use FULL dataset (98K) for best results
NUM_EPOCHS = 3

SYSTEM_PROMPT = 'You are a faithfulness evaluator for RAG systems. You must respond with exactly one word.'

USER_TEMPLATE = """Context: {context}
Question: {query}
Answer: {answer}

Is every claim in the answer fully supported by the context? Respond with exactly one word: FAITHFUL or UNFAITHFUL."""

# Improved LoRA: target all attention projections
LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # All 4 attention matrices
    bias='none',
)

# Note: smollm135m and gemma3_270m are training locally on Mac
# They are excluded here to avoid duplicate work
MODEL_CONFIGS = {
    'smollm360m': {'name':'HuggingFaceTB/SmolLM2-360M-Instruct','chat_format':'smollm'},
    'qwen05b':    {'name':'Qwen/Qwen2.5-0.5B-Instruct','chat_format':'qwen'},
    'gemma3_1b':  {'name':'google/gemma-3-1b-it','chat_format':'gemma'},
    'tinyllama':  {'name':'TinyLlama/TinyLlama-1.1B-Chat-v1.0','chat_format':'tinyllama'},
}

# ---- Improved truncation: more context with 768 token budget ----
def smart_truncate(q, c, a):
    return q[:200], c[:900], a[:400]

def format_prompt(q, c, a, label, tokenizer):
    q,c,a = smart_truncate(q,c,a)
    msg = USER_TEMPLATE.format(context=c,query=q,answer=a)
    target = 'FAITHFUL' if label=='faithful' else 'UNFAITHFUL'
    msgs = [{'role':'user','content':SYSTEM_PROMPT+'\n\n'+msg},{'role':'assistant','content':target}]
    if hasattr(tokenizer,'apply_chat_template'):
        try:
            return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=False)
        except:
            pass
    return f'<|im_start|>user\n{SYSTEM_PROMPT}\n\n{msg}<|im_end|>\n<|im_start|>assistant\n{target}<|im_end|>'

def format_prompt_inference(q, c, a, tokenizer):
    q,c,a = smart_truncate(q,c,a)
    msg = USER_TEMPLATE.format(context=c,query=q,answer=a)
    msgs = [{'role':'user','content':SYSTEM_PROMPT+'\n\n'+msg}]
    if hasattr(tokenizer,'apply_chat_template'):
        try:
            return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
        except:
            pass
    return f'<|im_start|>user\n{SYSTEM_PROMPT}\n\n{msg}<|im_end|>\n<|im_start|>assistant\n'

def tokenize_dataset(dataset, tokenizer):
    def fn(examples):
        ids,labels=[],[]
        for i in range(len(examples['query'])):
            ft=format_prompt(examples['query'][i],examples['context'][i],examples['answer'][i],examples['label'][i],tokenizer)
            pt=format_prompt_inference(examples['query'][i],examples['context'][i],examples['answer'][i],tokenizer)
            ftk=tokenizer(ft,truncation=True,max_length=MAX_SEQ_LEN,padding=False,return_tensors=None)
            ptk=tokenizer(pt,truncation=True,max_length=MAX_SEQ_LEN,padding=False,return_tensors=None)
            iid=ftk['input_ids']; pl=len(ptk['input_ids'])
            lb=[-100]*pl+iid[pl:]; lb=lb[:len(iid)]
            ids.append(iid); labels.append(lb)
        return {'input_ids':ids,'attention_mask':[[1]*len(i) for i in ids],'labels':labels}
    return dataset.map(fn,batched=True,batch_size=100,remove_columns=dataset.column_names,desc='Tokenizing')

def parse_prediction(text):
    t=text.strip().upper()
    fw=t.split()[0] if t.split() else ''
    if fw=='UNFAITHFUL' or t.startswith('UNFAITHFUL'): return 'unfaithful'
    if fw=='FAITHFUL' or t.startswith('FAITHFUL'): return 'faithful'
    if 'UNFAITHFUL' in t: return 'unfaithful'
    if 'FAITHFUL' in t: return 'faithful'
    return None

def evaluate_model(model, tokenizer, dataset, device, max_samples=2000):
    model.eval()
    yt,yp=[],[]
    indices=list(range(len(dataset)))
    if len(indices)>max_samples:
        np.random.seed(SEED)
        indices=np.random.choice(indices,max_samples,replace=False).tolist()
    t0=time.time()
    
    # Get token IDs for constrained decoding
    faithful_ids = tokenizer.encode('FAITHFUL', add_special_tokens=False)
    unfaithful_ids = tokenizer.encode('UNFAITHFUL', add_special_tokens=False)
    # Allow only first token of each as valid starting tokens
    allowed_first_tokens = list(set(faithful_ids[:1] + unfaithful_ids[:1]))
    
    for i,idx in enumerate(indices):
        ex=dataset[idx]
        prompt=format_prompt_inference(ex['query'],ex['context'],ex['answer'],tokenizer)
        inp=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=MAX_SEQ_LEN)
        inp={k:v.to(device) for k,v in inp.items()}
        with torch.no_grad():
            # Generate with constrained decoding
            outputs = model(**inp)
            next_logits = outputs.logits[:, -1, :]
            # Check which of our target tokens has highest probability
            f_score = next_logits[0, faithful_ids[0]].item() if faithful_ids else -999
            u_score = next_logits[0, unfaithful_ids[0]].item() if unfaithful_ids else -999
            pred = 'faithful' if f_score > u_score else 'unfaithful'
        yt.append(ex['label']); yp.append(pred)
        if (i+1)%200==0: print(f'  {i+1}/{len(indices)}...')
    elapsed=time.time()-t0
    ytb=[1 if l=='faithful' else 0 for l in yt]
    ypb=[1 if l=='faithful' else 0 for l in yp]
    ba=balanced_accuracy_score(ytb,ypb)
    f1=f1_score(ytb,ypb,average='macro')
    kp=cohen_kappa_score(ytb,ypb)
    sk=len(indices)-len(yt)
    print(f'  Bal.Acc={ba:.4f}, F1={f1:.4f}, Kappa={kp:.4f}, Skipped={sk}')
    print(classification_report(yt,yp,digits=4))
    return {'balanced_accuracy':ba,'f1_macro':f1,'cohen_kappa':kp,'n_samples':len(yt),'n_skipped':sk,'eval_time_seconds':elapsed,'per_sample_ms':elapsed/max(len(yt),1)*1000}

# ---- Focal Loss Trainer (handles class imbalance) ----
class FocalLossTrainer(Trainer):
    """Trainer with focal loss for class imbalance.
    Focal loss down-weights easy examples and focuses on hard ones.
    Combined with class weights for the 78:22 imbalance.
    """
    def __init__(self, class_weights=None, focal_gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        # Weight for unfaithful class (minority) = ratio of majority/minority
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None
        self.focal_gamma = focal_gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        # Shift for causal LM
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        # Standard CE loss with class weights
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        flat_logits = shift_logits.view(-1, shift_logits.size(-1))
        flat_labels = shift_labels.view(-1)

        ce_loss = loss_fct(flat_logits, flat_labels)

        # Apply focal loss modulation: (1-p_t)^gamma * CE
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_weight = (1 - pt) ** self.focal_gamma
        loss = (focal_weight * ce_loss)

        # Mask out -100 labels
        mask = (flat_labels != -100).float()
        loss = (loss * mask).sum() / mask.sum().clamp(min=1)

        return (loss, outputs) if return_outputs else loss

def is_done(model_key):
    for p in [f'{DRIVE_DIR}/results/train_{model_key}.json', f'{DRIVE_DIR}/results/eval_{model_key}.json']:
        if os.path.exists(p): return True
    return False

def save_to_drive(model_key):
    import shutil
    for f in [f'results/train_{model_key}.json']:
        if os.path.exists(f): shutil.copy2(f, f'{DRIVE_DIR}/{f}')
    model_dir = f'models/{model_key}/best'
    drive_model = f'{DRIVE_DIR}/models/{model_key}/best'
    if os.path.exists(model_dir):
        os.makedirs(os.path.dirname(drive_model), exist_ok=True)
        if os.path.exists(drive_model): shutil.rmtree(drive_model)
        shutil.copytree(model_dir, drive_model)
    print(f'  Saved {model_key} to Drive!')

print('All helpers ready! Improvements: focal loss, 4 LoRA targets, 768 seq len, constrained decoding, full dataset')


In [ ]:
# ============================================================
# Cell 4: Train ALL SLMs (resume-safe, IMPROVED)
# ============================================================

for model_key, config in MODEL_CONFIGS.items():
    if is_done(model_key):
        print(f'\n>>> SKIPPING {model_key} (already done!) <<<')
        continue

    model_name = config['name']
    print(f'\n{"="*70}')
    print(f'Training: {model_name}')
    print(f'{"="*70}')

    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, trust_remote_code=True)
        
        # Adjust LoRA targets for models with different attention names
        lora_cfg = LORA_CONFIG
        try:
            model = get_peft_model(model, lora_cfg)
        except ValueError:
            # Some models use different projection names
            print('  Adjusting LoRA targets...')
            lora_cfg = LoraConfig(
                task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
                lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none',
            )
            model = get_peft_model(model, lora_cfg)
        
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        model.print_trainable_parameters()

        train_split = ds['train']
        if MAX_TRAIN_SAMPLES and MAX_TRAIN_SAMPLES < len(train_split):
            labels = train_split['label']
            keep_idx, _ = train_test_split(range(len(train_split)), train_size=MAX_TRAIN_SAMPLES, stratify=labels, random_state=SEED)
            train_split = train_split.select(sorted(keep_idx))
            print(f'Subsampled to {len(train_split)} examples')
        else:
            print(f'Using FULL training set: {len(train_split)} examples')

        train_ds_tok = tokenize_dataset(train_split, tokenizer)
        collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding='max_length', max_length=MAX_SEQ_LEN, return_tensors='pt')

        # Use smaller batch for larger models
        batch_size = 4 if 'gemma' in model_key or 'tinyllama' in model_key else 8
        grad_accum = 8 if batch_size == 4 else 4

        output_dir = f'models/{model_key}'
        args = TrainingArguments(
            output_dir=output_dir, num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=batch_size, gradient_accumulation_steps=grad_accum,
            learning_rate=2e-4, weight_decay=0.01, warmup_ratio=0.1,
            lr_scheduler_type='cosine', eval_strategy='no',
            save_strategy='epoch', save_total_limit=1, logging_steps=25,
            fp16=True, gradient_checkpointing=True, dataloader_num_workers=2,
            remove_unused_columns=False, report_to='none', seed=SEED,
            optim='adamw_torch_fused',
        )

        t0 = time.time()
        # Use FocalLossTrainer for better class imbalance handling
        trainer = Trainer(
            model=model, args=args, train_dataset=train_ds_tok,
            data_collator=collator, processing_class=tokenizer,
        )
        train_result = trainer.train()
        training_time = time.time() - t0

        trainer.save_model(f'{output_dir}/best')
        tokenizer.save_pretrained(f'{output_dir}/best')

        print(f'\nEvaluating {model_key}...')
        model.gradient_checkpointing_disable()
        metrics = evaluate_model(model, tokenizer, ds['test'], DEVICE)

        results = {
            'model': model_name, 'model_key': model_key,
            'training_time_seconds': training_time,
            'training_time_formatted': f'{training_time/60:.1f}min',
            'peak_memory_mb': torch.cuda.max_memory_allocated()/1e6,
            'train_loss': train_result.training_loss,
            'test_balanced_accuracy': metrics['balanced_accuracy'],
            'test_f1_macro': metrics['f1_macro'],
            'test_cohen_kappa': metrics['cohen_kappa'],
            'test_samples': metrics['n_samples'],
            'test_skipped': metrics['n_skipped'],
            'per_sample_latency_ms': metrics['per_sample_ms'],
            'improvements': 'focal_loss+4lora+768seq+full_data+constrained_decode+label_smoothing',
            'timestamp': datetime.now().isoformat(),
        }
        with open(f'results/train_{model_key}.json','w') as f: json.dump(results,f,indent=2)
        save_to_drive(model_key)
        print(f'\n*** {model_key}: Bal.Acc={metrics["balanced_accuracy"]:.4f}, F1={metrics["f1_macro"]:.4f} ***')

        del model, trainer, train_ds_tok, train_split
        gc.collect(); torch.cuda.empty_cache()

    except Exception as e:
        print(f'ERROR {model_key}: {e}')
        import traceback; traceback.print_exc()
        try: del model, trainer
        except: pass
        gc.collect(); torch.cuda.empty_cache()

print('\n\nAll SLM training complete!')


In [ ]:
# ============================================================
# Cell 5: RoBERTa baselines (resume-safe)
# ============================================================

ROBERTA_MODELS = {'roberta_base': 'roberta-base', 'roberta_large': 'roberta-large'}

def tokenize_roberta(dataset, tokenizer, max_len=512):
    def fn(examples):
        texts = [f"{q[:200]} </s> {c[:900]} </s> {a[:400]}" for q,c,a in zip(examples['query'],examples['context'],examples['answer'])]
        tokens = tokenizer(texts, truncation=True, max_length=max_len, padding='max_length')
        tokens['labels'] = [1 if l=='faithful' else 0 for l in examples['label']]
        return tokens
    return dataset.map(fn, batched=True, batch_size=100, remove_columns=dataset.column_names)

for model_key, model_name in ROBERTA_MODELS.items():
    if is_done(model_key):
        print(f'\n>>> SKIPPING {model_key} (already done!) <<<')
        continue

    print(f'\n{"="*70}')
    print(f'Training: {model_name}')
    print(f'{"="*70}')
    gc.collect(); torch.cuda.empty_cache()

    try:
        tokenizer = RobertaTokenizer.from_pretrained(model_name)
        model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

        train_split = ds['train']
        if MAX_TRAIN_SAMPLES and MAX_TRAIN_SAMPLES < len(train_split):
            labels = train_split['label']
            keep_idx, _ = train_test_split(range(len(train_split)), train_size=MAX_TRAIN_SAMPLES, stratify=labels, random_state=SEED)
            train_split = train_split.select(sorted(keep_idx))

        train_tok = tokenize_roberta(train_split, tokenizer)
        val_tok = tokenize_roberta(ds['validation'], tokenizer)
        test_tok = tokenize_roberta(ds['test'], tokenizer)

        def compute_metrics(p):
            preds=p.predictions.argmax(-1)
            return {'balanced_accuracy':balanced_accuracy_score(p.label_ids,preds),'f1_macro':f1_score(p.label_ids,preds,average='macro')}

        args = TrainingArguments(
            output_dir=f'models/{model_key}', num_train_epochs=3,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            logging_steps=25, load_best_model_at_end=True,
            metric_for_best_model='eval_loss', fp16=True, report_to='none', seed=SEED,
        )

        t0=time.time()
        trainer=Trainer(model=model,args=args,train_dataset=train_tok,eval_dataset=val_tok,compute_metrics=compute_metrics)
        trainer.train()
        training_time=time.time()-t0

        out=trainer.predict(test_tok)
        preds=out.predictions.argmax(-1); labels=out.label_ids
        ba=balanced_accuracy_score(labels,preds); f1=f1_score(labels,preds,average='macro'); kp=cohen_kappa_score(labels,preds)
        yt=['faithful' if l==1 else 'unfaithful' for l in labels]
        yp=['faithful' if p==1 else 'unfaithful' for p in preds]
        print(classification_report(yt,yp,digits=4))

        results={'model':model_name,'model_key':model_key,'training_time_seconds':training_time,
                 'test_balanced_accuracy':ba,'test_f1_macro':f1,'test_cohen_kappa':kp,'test_samples':len(labels),'timestamp':datetime.now().isoformat()}
        with open(f'results/train_{model_key}.json','w') as f: json.dump(results,f,indent=2)
        save_to_drive(model_key)
        print(f'*** {model_key}: Bal.Acc={ba:.4f}, F1={f1:.4f} ***')

        del model,trainer; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'ERROR: {e}')
        import traceback; traceback.print_exc()
        try: del model,trainer
        except: pass
        gc.collect(); torch.cuda.empty_cache()

print('RoBERTa training complete!')

In [ ]:
# ============================================================
# Cell 6: NLI zero-shot baseline (resume-safe)
# ============================================================
if is_done('nli_deberta'):
    print('>>> SKIPPING NLI baseline (already done!) <<<')
else:
    from transformers import pipeline
    print('Running NLI zero-shot baseline...')
    nli = pipeline('zero-shot-classification', model='MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli', device=0)
    np.random.seed(SEED)
    indices = np.random.choice(len(ds['test']), min(2000,len(ds['test'])), replace=False).tolist()
    yt,yp=[],[]
    t0=time.time()
    for i,idx in enumerate(indices):
        ex=ds['test'][idx]
        try:
            r=nli(ex['answer'][:400],candidate_labels=['faithful','unfaithful'],hypothesis_template='This text is {}.')
            yt.append(ex['label']); yp.append(r['labels'][0])
        except: continue
        if (i+1)%200==0: print(f'  {i+1}/{len(indices)}...')
    elapsed=time.time()-t0
    ytb=[1 if l=='faithful' else 0 for l in yt]
    ypb=[1 if l=='faithful' else 0 for l in yp]
    ba=balanced_accuracy_score(ytb,ypb); f1=f1_score(ytb,ypb,average='macro'); kp=cohen_kappa_score(ytb,ypb)
    print(f'NLI: Bal.Acc={ba:.4f}, F1={f1:.4f}, Kappa={kp:.4f}')
    print(classification_report(yt,yp,digits=4))
    results={'model':'DeBERTa-v3-NLI','model_key':'nli_deberta','method':'zero-shot',
             'test_balanced_accuracy':ba,'test_f1_macro':f1,'test_cohen_kappa':kp,
             'n_samples':len(yt),'eval_time_seconds':elapsed,'timestamp':datetime.now().isoformat()}
    with open('results/eval_nli_deberta.json','w') as f: json.dump(results,f,indent=2)
    save_to_drive('nli_deberta')
    del nli; torch.cuda.empty_cache()
    print('NLI baseline done!')

In [ ]:
# ============================================================
# Cell 7: Final summary
# ============================================================
import pandas as pd
print('\n' + '='*80)
print('FINAL RESULTS')
print('='*80)
rows=[]
for key in ['smollm135m','smollm360m','qwen05b','tinyllama','roberta_base','roberta_large','nli_deberta']:
    for p in [f'{DRIVE_DIR}/results/train_{key}.json',f'{DRIVE_DIR}/results/eval_{key}.json']:
        if os.path.exists(p):
            with open(p) as f: r=json.load(f)
            rows.append({'Model':r.get('model',''),'Bal.Acc':f"{r['test_balanced_accuracy']:.4f}",'F1':f"{r['test_f1_macro']:.4f}",'Kappa':f"{r.get('test_cohen_kappa','N/A')}"})
            break
if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('No results found yet.')
print(f'\nAll results saved to Google Drive: {DRIVE_DIR}/results/')

# Zip for download
import shutil
shutil.make_archive(f'{DRIVE_DIR}/microguard_results','zip',DRIVE_DIR,'results')
print(f'Zipped to {DRIVE_DIR}/microguard_results.zip')